# einops-repeat — ex2: per-token weight → per-feature weight

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `einops-repeat`. When a test cell passes, your progress is reported back to your account.

**What you'll practice.** Five `einops.repeat` patterns that ramp from new-axis broadcast → per-token-to-per-feature → vertical stretch → horizontal tile → 2×2 nearest-neighbor upsample. Read the docstring, fill the function body, run the test cell. The solution sits in the collapsed `<details>` block below each exercise.

**Per-exercise structure** (Doughty et al. ACE 2024 — `[Bloom level] + [LO] + [Keywords] + [KCs]`):
Each exercise begins with a yaml block stating its Bloom cognitive level, learning objective, keywords, and the knowledge components (KCs) it targets. This makes the cognitive demand explicit instead of buried.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import torch.nn.functional as F
import einops
from einops import repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Einops: Repeat` subtopic.
You can copy the token from your Delta Drills account page.

This drill exercises the **atom `einops-repeat`**, which bridges to the bank subtopic `Einops: Repeat` for EWMA state. Completing all 5 exercises triggers a single `arena-rating` beacon at the end of the notebook.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "einops-repeat"
DD_SUBTOPIC = "Einops: Repeat"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

# Track which exercises passed in this session.
_dd_passed = set()

## einops.repeat — quick refresher

`repeat(tensor, pattern, **axes_lengths)` introduces new axes or stretches existing ones:
1. **New axis** — `'c h w -> b c h w'` with `b=4` broadcasts across a new batch dim.
2. **Trailing axis** — `'b t -> b t d'` with `d=64` materializes a per-token weight at per-feature width.
3. **Stretch (nearest-neighbor)** — `'h w -> (h r) w'` with `r=2` makes each row appear twice in a block.
4. **Tile** — `'h w -> h (r w)'` with `r=2` concatenates two copies of every row.

Difference between **stretch** `(h r)` and **tile** `(r h)`: the factor written first varies slower. `(h r)` puts source row 0 at positions `0..r-1`; `(r h)` puts source row 0 at positions `0, h, 2h, ...`.

### Exercise 2 — per-token weight → per-feature weight

> ```yaml
> Difficulty: 🔴⚪⚪⚪⚪
> Bloom level: Apply
> LO: Apply repeat to add a trailing feature axis that lets a per-token weight broadcast against a per-token-per-feature tensor.
> Keywords: match-shape, trailing-axis, attention-mask
> ```

**KCs targeted:** `repeat-match-broadcast-shape`

Implement `ex2_per_token_to_per_feature(w, d)` so each per-token weight is materialized across `d` feature columns.

Input shape: `(b, t)`. Output shape: `(b, t, d)`. Every `y[b, t, :]` should equal `w[b, t]`.

This is the shape you need when you want to scale a feature tensor of shape `(b, t, d)` by a per-token weight.

In [ ]:
def ex2_per_token_to_per_feature(w: Tensor, d: int) -> Tensor:
    return repeat(w, 'b t -> b t d', d=d)


<details><summary>Solution</summary>

```python
def ex2_per_token_to_per_feature(w: Tensor, d: int) -> Tensor:
    return repeat(w, 'b t -> b t d', d=d)
```
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',  # single-exercise standalone — neutral signal
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()